# Leukemia Detection — VGG16 Transfer Learning (No Augmentation)

In [ ]:
# 1. Imports

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## Merge fold_0 + fold_1 → Training | fold_2 → Testing

In [ ]:
# 2. Merge fold_0 and fold_1 into a single training directory

fold_0 = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_0/fold_0'
fold_1 = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_1/fold_1'
test_dir = '/kaggle/input/cnmc-leukemia/C-NMC_Leukemia/fold_2/fold_2'

merged_train_dir = '/kaggle/working/merged_train'

# Get class names from fold_0
class_names = sorted([d for d in os.listdir(fold_0) if os.path.isdir(os.path.join(fold_0, d))])

# Create merged directory and copy images
for cls in class_names:
    os.makedirs(os.path.join(merged_train_dir, cls), exist_ok=True)

    # Copy from fold_0
    src_0 = os.path.join(fold_0, cls)
    for img in os.listdir(src_0):
        shutil.copy2(os.path.join(src_0, img), os.path.join(merged_train_dir, cls, img))

    # Copy from fold_1
    src_1 = os.path.join(fold_1, cls)
    for img in os.listdir(src_1):
        shutil.copy2(os.path.join(src_1, img), os.path.join(merged_train_dir, cls, img))

# Count
print("Merged Training Set:")
for cls in class_names:
    n = len(os.listdir(os.path.join(merged_train_dir, cls)))
    print(f"  {cls}: {n} images")

print("\nTest Set:")
for cls in sorted(os.listdir(test_dir)):
    cls_path = os.path.join(test_dir, cls)
    if os.path.isdir(cls_path):
        print(f"  {cls}: {len(os.listdir(cls_path))} images")

## Data Preprocessing (No Augmentation)

In [ ]:
# 3. Data generators — rescaling ONLY, no augmentation

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.15   # 85% train, 15% val from the merged set
)

test_datagen = ImageDataGenerator(rescale=1.0/255)

train_gen = datagen.flow_from_directory(
    merged_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    subset='training'
)

val_gen = datagen.flow_from_directory(
    merged_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
    subset='validation'
)

test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"\nClass mapping: {train_gen.class_indices}")
print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Test samples: {test_gen.samples}")

In [ ]:
# Visualize samples
images, labels = next(train_gen)
class_labels = list(train_gen.class_indices.keys())

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(class_labels[int(labels[i])], fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('Sample Training Images (No Augmentation)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 1 — Feature Extraction (Frozen VGG16)

In [ ]:
# 4. Load VGG16 and build model

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
# 5. Compile for Phase 1

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# 6. Phase 1 Training

print("PHASE 1: Feature Extraction (Frozen VGG16 Base)")
print("=" * 50)

history_phase1 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)

In [ ]:
# Phase 1 Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_phase1.history['accuracy'], label='Train')
ax1.plot(history_phase1.history['val_accuracy'], label='Val')
ax1.set_title('Phase 1 — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_phase1.history['loss'], label='Train')
ax2.plot(history_phase1.history['val_loss'], label='Val')
ax2.set_title('Phase 1 — Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Best Phase 1 Val Accuracy: {max(history_phase1.history['val_accuracy'])*100:.2f}%")

## Phase 2 — Fine-Tuning (Unfreeze Block 5)

In [ ]:
# 7. Unfreeze last conv block (Block 5)

base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

for i, layer in enumerate(base_model.layers):
    status = "TRAINABLE" if layer.trainable else "FROZEN"
    print(f"  Layer {i:2d}: {layer.name:20s} -> {status}")

In [ ]:
# 8. Recompile with lower learning rate

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
# 9. Phase 2 Training

print("PHASE 2: Fine-Tuning (Block 5 + Classifier)")
print("=" * 50)

history_phase2 = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)

In [ ]:
# Phase 2 Results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_phase2.history['accuracy'], label='Train', color='#e74c3c')
ax1.plot(history_phase2.history['val_accuracy'], label='Val', color='#3498db')
ax1.set_title('Phase 2 — Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history_phase2.history['loss'], label='Train', color='#e74c3c')
ax2.plot(history_phase2.history['val_loss'], label='Val', color='#3498db')
ax2.set_title('Phase 2 — Loss', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_p1 = max(history_phase1.history['val_accuracy'])
best_p2 = max(history_phase2.history['val_accuracy'])
print(f"Best Phase 2 Val Accuracy: {best_p2*100:.2f}%")
print(f"Improvement over Phase 1: {(best_p2 - best_p1)*100:+.2f}%")

## Evaluation

In [ ]:
# 10. Evaluate on test set (fold_2)

test_loss, test_acc = model.evaluate(test_gen)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
# 11. Confusion Matrix

test_gen.reset()
y_pred = (model.predict(test_gen) > 0.5).astype(int).flatten()
y_true = test_gen.classes
class_labels = list(test_gen.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, annot_kws={'size': 16})
plt.title('Confusion Matrix — VGG16 (No Augmentation)', fontsize=16, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# 12. Classification Report

print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
# 13. Visualize predictions

test_gen.reset()
images, labels = next(test_gen)
predictions = model.predict(images)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    true_label = class_labels[int(labels[i])]
    pred_label = class_labels[int(predictions[i] > 0.5)]
    confidence = predictions[i][0] if predictions[i] > 0.5 else 1 - predictions[i][0]
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.1%})",
                fontsize=10, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 14. Save model

model.save('vgg16_leukemia_noaug_final.keras')
print("Model saved as 'vgg16_leukemia_noaug_final.keras'")